# Experiment 3 — class-imbalance robustness (PD)

Metric vs **minority-class proportion** (nested minority removal). One line per method, averaged over the folds of every included dataset (linear x), shown as raw points, **moving average**, and **relative** to each method's own best. Watch **AP_normalized** — the prevalence-corrected metric. Figures → `figures/experiment3/`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, method_ranking_bars, learning_curve, imbalance_curve,
    metric_boxplots, metric_bars, compute_time_bars, compute_time_boxplot,
    rank_heatmap, rank_boxplots, hpo_improvement_bars, runtime_performance_scatter,
    pd_summary_text, lgd_summary_text,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment3')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
df = load_summary(SUMMARY_DIR, experiment='experiment3', task='pd', aggregated=False)
print(f'{df["method"].nunique()} methods, {df["dataset"].nunique()} datasets, '
      f'{df["sweep_value"].nunique()} sweep points')

## AUC (points, moving average, relative)

In [ ]:
imbalance_curve(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)
imbalance_curve(df, 'AUC', task_name='PD', smooth=True,   out_dir=FIGURES_DIR)
imbalance_curve(df, 'AUC', task_name='PD', relative=True, out_dir=FIGURES_DIR)

## AP_normalized (prevalence-corrected)

In [ ]:
imbalance_curve(df, 'AP_normalized', task_name='PD', out_dir=FIGURES_DIR)
imbalance_curve(df, 'AP_normalized', task_name='PD', smooth=True, out_dir=FIGURES_DIR)

## Degradation: AUC at the easiest minus the hardest setting

In [ ]:
import pandas as _pd
g = df.groupby(['method','sweep_value'])['metric.AUC'].mean().reset_index()
drop = {m: gg.sort_values('sweep_value').iloc[-1]['metric.AUC']
            - gg.sort_values('sweep_value').iloc[0]['metric.AUC']
        for m, gg in g.groupby('method')}
display(_pd.Series(drop, name='AUC(max minority) - AUC(min minority)').sort_values(ascending=False))

## Summary (at the most balanced setting)

In [ ]:
full = df[df['sweep_value'] == df['sweep_value'].max()]
pd_summary_text(full, task_name='Experiment 3 — PD @ max minority')